# Choosing accuracy and performance settings
This notebook measures real map accuracy, runtime, and memory on the current machine. It explains the independent numerical controls rather than prescribing one universal configuration. CUDA systems use strict Triton execution so a fallback cannot be mistaken for an accelerated timing. Other systems use compiled Torch when a compatible toolchain is available and otherwise fall back to eager Torch.

## What the controls mean
- **`rays` ($N$)** sets the base lens-plane cell or ray budget.
- **`scout_ratio` ($k$)** coarsens only tiled source-region discovery. It does not reduce the retained fine-cell resolution.
- **`refinement` ($r$)** adds genuine lens-equation samples inside retained IPM cells.
- **`virtual_refinement` ($v$)** increases the polygon representation by interpolation and must satisfy $v\ge r$.
- The **local-exact/far-field approximation** evaluates nearby point masses exactly and represents the far field with a complex Taylor expansion.
- **Temporal batching and scout reuse** affect dynamic throughput. Their choices are recorded in result metadata.

Always compare configurations on the actual lens and source scales of interest. A small tutorial field is useful for learning the workflow, not for establishing universal error bounds.

In [ ]:
import math
import json
from pathlib import Path
import platform
import numpy as np
import torch
from matplotlib import pyplot as plt
import microcaustics as mc
import microcaustics.plotting as mcp

plt.rcParams.update(mcp.publication_style())
OUTPUT_DIRECTORY = Path('accuracy_performance_output')
OUTPUT_DIRECTORY.mkdir(exist_ok=True)
capabilities = mc.RuntimeCapabilities.detect()
use_cuda = capabilities.cuda_available and capabilities.triton_importable
runtime_config = mc.RuntimeConfig(
    device='cuda' if use_cuda else 'cpu',
    backend='triton' if use_cuda else 'auto',
    dtype=torch.float32,
    strict_backend=use_cuda,
    memory_fraction=0.80,
)
runtime = mc.resolve_runtime(runtime_config)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('Selected device/backend:', runtime.device, runtime.backend.value)
print('GPU:', capabilities.gpu_name)
print('CUDA:', capabilities.cuda_version, 'Triton:', capabilities.triton_importable)


In [ ]:
distances = mc.LensingDistances.from_redshifts(0.0395, 1.695)
macro = mc.MacroLens(
    convergence=0.391, shear=0.391,
    shear_angle_deg=141.73, smooth_matter_fraction=0.0,
)
lens_diameter_uas = 754.11
lens_region = mc.PlaneRegion((lens_diameter_uas, lens_diameter_uas))
aperture_radius_uas = 0.5 * lens_diameter_uas
population = mc.StellarPopulation.salpeter(
    mean_mass_solar=0.3, mass_ratio=100.0,
    kinematics=mc.SkyProjectedKinematics(
        ra_deg=340.126125, dec_deg=3.358611,
        stellar_dispersion_km_s=170.0,
        peculiar_velocity_dispersion_km_s=235.0,
        include_cmb_dipole=True, seed=0,
    ),
)
mass_function = population.mass_function
field = population.realize(
    mc.StellarAperture(aperture_radius_uas), macro, distances,
    seed=0, dtype=torch.float64,
)
source_grid = mc.PlaneGrid((1024, 1024), (9.3819, 9.3819))
system = mc.MicrolensingSystem(
    macro=macro, distances=distances, stars=field, source_grid=source_grid,
    lens_region=lens_region, duration_days=30.0, runtime=runtime_config,
)
simulation = system.realize().simulation  # direct timing access for this benchmark
far_field = mc.production_ipm_config().far_field_approx

# Calculate a v=8 reference one decade above the fitted candidate range.
# The separate publication workflow uses the partitioned 10^9-cell gold map
# when extending the convergence figure through 10^8 candidate cells.
reference_rays = 100_000_000 if use_cuda else 4_000_000
reference_method = mc.production_ipm_config(
    rays=reference_rays,
    scout_ratio=1,
    virtual_refinement=8,
    cell_chunk_size=524_288 if use_cuda else 65_536,
)
reference_map = simulation.magnification_map(
    lens_region, source_grid, method=reference_method,
)
print(
    f"Live reference: N={reference_rays:,}, "
    f"rasterizer={reference_map.metadata.get('rasterizer')}"
)


## A small convergence and timing sweep
`benchmark_callable` reports the first call separately because it may contain compilation, allocator initialization, and cache population. Because one warmed map is very fast, each plotted timing repeat contains a fixed bundle of maps. The displayed value is the median bundle time divided by its map count, in seconds per map. Both dashed lines are direct log--log power-law fits over the complete displayed $10^6$--$10^7$ range. Their fitted exponents are computed when the notebook runs rather than inserted as fixed tutorial values. Map comparisons retain constant normalization bias. Both panels are generated live. The accuracy panel compares each Q2237 image-B map through $10^7$ cells with a $10^8$-cell, $v=8$ reference. The separate publication workflow uses its partitioned $10^9$-cell gold map when extending candidate maps through $10^8$ cells. The runtime panel measures the current machine with the production $k=1$, $r=2$, $v=4$ static-map settings.


In [ ]:
rows = []
# One production map is fast enough that allocator and host-timer jitter can
# obscure the scaling. Time fixed bundles and divide by their map count.
timing_maps_per_repeat = 4 if use_cuda else 2
ray_budgets = (
    (1_000_000, 2_500_000, 5_000_000, 10_000_000)
    if use_cuda else (100_000, 250_000, 500_000, 1_000_000)
)
for rays in ray_budgets:
    method = mc.production_ipm_config(
        rays=rays, scout_ratio=1,
        cell_chunk_size=524_288 if use_cuda else 65_536,
        far_field_approx=far_field,
    )
    def map_bundle(method=method):
        result = None
        for _ in range(timing_maps_per_repeat):
            result = simulation.magnification_map(lens_region, source_grid, method=method)
        return result
    measured = mc.benchmark_callable(
        map_bundle, warmup=1, repeats=5,
        synchronize=simulation.runtime.synchronize,
    )
    result = measured.first_result
    comparison = mc.compare_magnification_maps(
        result.values, reference_map.values,
    )
    rows.append({
        "rays": rays,
        "first_bundle_s": measured.first_call_seconds,
        "steady_s": measured.median_steady_seconds / timing_maps_per_repeat,
        "steady_std_s": measured.steady_standard_deviation_seconds / timing_maps_per_repeat,
        "nrmse": comparison.fractional_nrmse,
        "rmse_mmag": comparison.rmse_mmag,
        "rasterizer": result.metadata.get("rasterizer"),
        "actual_rays": result.metadata.get("actual_rays"),
        "selected_cells": result.metadata.get("selected_cells"),
        "construction_batches": result.metadata.get("construction_batches"),
    })

for row in rows:
    print(row)

figure, axes = plt.subplots(1, 2, figsize=(9.0, 3.6))
ray_values = [row["rays"] for row in rows]
nrmse_values = [row["nrmse"] for row in rows]
axes[0].loglog(ray_values, nrmse_values, "o-", label="Tiled IPM")
fit_count = len(rows)
slope, intercept = np.polyfit(
    np.log10(ray_values[:fit_count]), np.log10(nrmse_values[:fit_count]), 1
)
fit_x = np.asarray(ray_values[:fit_count], dtype=float)
axes[0].loglog(
    fit_x, 10.0 ** intercept * fit_x ** slope, '--',
    color='0.25', label=fr"full-range fit, $p={slope:.2f}$",
)
axes[0].set(xlabel="Base-cell budget $N$", ylabel="Fractional NRMSE")
runtime_values = np.asarray([row["steady_s"] for row in rows])
runtime_errors = np.asarray([row["steady_std_s"] for row in rows])
axes[1].errorbar(
    ray_values, runtime_values, yerr=runtime_errors, fmt="s-",
    capsize=2.5, label="Static tiled IPM ($k=1$)",
)
runtime_fit_mask = np.ones(len(ray_values), dtype=bool)
runtime_slope, runtime_intercept = np.polyfit(
    np.log10(np.asarray(ray_values, dtype=float)[runtime_fit_mask]),
    np.log10(runtime_values[runtime_fit_mask]), 1,
)
runtime_fit_x = np.geomspace(float(min(ray_values)), float(max(ray_values)), 200)
axes[1].loglog(
    runtime_fit_x, 10.0**runtime_intercept * runtime_fit_x**runtime_slope,
    '--', color='0.25', label=fr"full-range fit, $p={runtime_slope:.2f}$",
)
axes[1].set(xlabel="Base-cell budget $N$", ylabel="Warmed runtime per map [s]")
axes[1].set_yticks([0.01, 0.02, 0.05, 0.10, 0.20])
axes[1].set_yticklabels(['0.01', '0.02', '0.05', '0.10', '0.20'])
if use_cuda:
    for ax in axes:
        ax.set_xlim(1.0e6, 1.0e7)
for ax in axes:
    mcp.finish_axis(ax)
    ax.legend(frameon=True)
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / 'accuracy_and_performance_convergence.png', dpi=220, bbox_inches='tight', facecolor='white')
print(f'Live Q2237 B convergence slope: p={slope:.3f}')
print(
    f'Warmed static-map runtime at N={ray_values[-1]:,}: '
    f'{runtime_values[-1]:.4f} s/map; '
    f'full-range exponent p={runtime_slope:.3f}'
)


## Dynamic fusion and automatic work-size tuning
The next calculation uses moving lenses and a short map sequence. Tuning changes only lossless temporal and spatial work sizes. It never changes $N$, $k$, $r$, $v$, the physical lens, or the map grid. Production studies should tune using representative frame counts and then reuse the selected settings.

In [ ]:
schedule = mc.production_dynamic_config(
    tuning=mc.AutoTuningConfig(
        enabled=use_cuda,
        temporal_candidates=(2, 4),
        spatial_candidates=(32_768, 65_536),
        maximum_trial_frames=4,
        memory_fraction=0.80,
    ),
)
dynamic_method = mc.production_ipm_config(
    rays=10_000_000 if use_cuda else 1_000_000,
    cell_chunk_size=524_288 if use_cuda else 65_536,
)
if use_cuda:
    torch.cuda.reset_peak_memory_stats(simulation.runtime.device)
frames = list(simulation.dynamic_maps(
    lens_region, source_grid, [0.0, 10.0, 20.0, 30.0],
    method=dynamic_method, schedule=schedule,
))
last = frames[-1]
print('Generated frames:', len(frames))
print('Actual raster backend:', last.metadata.get('rasterizer'))
print('Selected temporal batch:', last.metadata.get('autotune_temporal_batch_size'))
print('Selected spatial chunk:', last.metadata.get('autotune_spatial_chunk_size'))
print('Approximate scout reuse:', last.metadata.get('dynamic_scout_reuse_approximate'))
peak_bytes = torch.cuda.max_memory_allocated(simulation.runtime.device) if use_cuda else 0
print('Peak CUDA tensor memory [GiB]:', peak_bytes / 1024**3)
figure, axes = mcp.plot_map_comparison(
    frames[0],
    frames[-1],
    residual="magnitude",
    scale_bar_uas=1.0,
    show_axes=False,
)
figure.tight_layout()


## Practical selection rules
1. Choose a map field and pixel scale from the physical source problem.
2. Increase $N$ until the chosen map or light-curve metric stabilizes.
3. Increase $r$ when finite-cell mapping artifacts remain. Increase $v$ when interpolation improves them without extra lens-equation samples.
4. Validate tiled IPM against full-field IPM on representative frames.
5. Validate the far-field approximation against direct deflections and validate scout reuse against `scout_refresh_frames=1`.
6. Tune batches and chunks only after numerical settings are fixed.
7. Report the actual backend, first-call policy, warmed timing, memory peak, and retained approximations.